# Analyse HELIOS Simulations Against Reference Voxel Maps

This notebook demonstrates the complete workflow for analyzing HELIOS simulation outputs and comparing them with reference voxel maps generated by `process_reference_single_obj.py`.

## Workflow Overview

**Step 1**: Extract valid rays from HELIOS simulation outputs and prepare reference voxel grids.  
**Step 2**: Trace rays through voxels and record intersection data (entry/exit points, classifications).  
**Step 3**: Calculate optical metrics (LAD, PAD, gap probability, G-function, normals/weights) and merge with reference data for validation.

# Step 1 - Configure Paths and Object IDs

Set the directory paths for HELIOS simulation outputs, reference mesh data, and output locations. Specify which object IDs in the HELIOS data correspond to leaf and wood materials.

## Path Configuration

**`project_dir`**: Base directory for your analysis. Used as the root for all subdirectories and to name output files.

**`helios_dir`**: Directory containing HELIOS simulation output files. Expected file structure:
- `leg{N}_points.xyz` — Ray hit coordinates for leg N (space-delimited: x, y, z, intensity, return#, ..., object_id, class, ...)
- `leg{N}_pulse.txt` — Pulse descriptor for leg N
- `leg{N}_fullwave.txt` — Full waveform data for leg N

Each leg N (000, 001, 002, etc.) represents a separate scanning position. The `object_id` column (typically column 8, unless `use_class=True`) is used to classify hits as leaf vs. wood based on your `leaf_object_ids` and `wood_object_ids` settings.

**`references_dir`**: Directory containing reference voxel maps generated by `process_reference_single_obj.py` from your 3D model. Expected file pattern:
- Files named like: `{model_name}_voxel_size_0.2.csv`, `{model_name}_voxel_size_0.5.csv`, etc.
- Each CSV contains ground-truth optical metrics computed from the reference mesh: voxel_id, LAD_ref, PAD_ref, G_leaf_ref, G_wood_ref, LIAD bins, clumping indices, etc.
- One CSV file per voxel size you want to analyze

**`results_dir`**: Output directory where final merged results are saved as CSV files. Format: `{project_name}_leg_{legs}_voxel_size_{size}.csv`. Created automatically if it doesn't exist.

**`valid_rays_dir`**: Working directory for intermediate Parquet files: valid rays extracted from HELIOS, and ray-voxel intersection data. Created automatically if it doesn't exist.

## Object Classification

**`use_class`**: Determines which column in HELIOS data identifies material type.
- `False` (default): Uses column 8 as `hitObject_id` — the geometry object ID assigned in HELIOS simulation
- `True`: Uses column 9 as `class` — a separate material class field

**`leaf_object_ids`**: List of object/class IDs in HELIOS data representing leaf geometry.
- Example: `[0]` or `[1]` depending on how your HELIOS scene was set up
- Must be mutually exclusive with `wood_object_ids`

**`wood_object_ids`**: List of object/class IDs in HELIOS data representing wood/stem geometry.
- Example: `[1]` if leaves are `[0]`, or `[2, 3]` for multiple wood components
- Must be mutually exclusive with `leaf_object_ids`

To verify your IDs are correct, run the classification verification cell next. It plots the first 1000 HELIOS points colored by your classification—green=leaf, brown=wood, blue=other.

In [ ]:
import os
from utils import test_helios_settings

# Set up the project directory
project_dir = '/path/to/your/project'                   # Update this path to your project directory
helios_dir = os.path.join(project_dir, 'helios')        # Change this to the actual path of your helios directory
references_dir = os.path.join(project_dir, 'references')
results_dir = os.path.join(project_dir, 'results')
valid_rays_dir = os.path.join(project_dir, 'valid_rays')

if not os.path.exists(helios_dir) or not os.path.exists(references_dir):
    raise FileNotFoundError("The specified directories do not exist. Please check the paths.")

if not os.path.exists(valid_rays_dir):
    os.makedirs(valid_rays_dir, exist_ok=True)

if not os.path.exists(results_dir):
    os.makedirs(results_dir, exist_ok=True)

use_class = False
leaf_object_ids = [0]
wood_object_ids = [1]

test_helios_settings(
    helios_dir=helios_dir,
    use_class=use_class,
    leaf_object_ids=leaf_object_ids,
    wood_object_ids=wood_object_ids,
    output_dir=project_dir,
)

## Step 2 - Extract and Filter Valid Rays from HELIOS Data

Convert HELIOS simulation outputs (XYZ points, pulse, fullwave data) into efficient Parquet files containing only valid rays. Also prepare reference voxel grids for each voxel size defined in the reference data.

**Inputs:** HELIOS simulation files from `helios_dir` + Reference CSVs from `references_dir`  
**Outputs:** Valid ray Parquet files in `valid_rays_dir` (one per leg)

### Data Structure

**INPUT:**
```
project_dir/
├── helios/                          (HELIOS simulation output)
│   ├── leg000_points.xyz            (ray hit coordinates)
│   ├── leg000_pulse.txt
│   ├── leg000_fullwave.txt
│   ├── leg001_points.xyz
│   ├── leg001_pulse.txt
│   ├── leg001_fullwave.txt
│   └── ... (additional legs)
│
└── references/                      (reference voxel maps from process_reference_single_obj.py)
    ├── {model}_voxel_size_0.2.csv
    ├── {model}_voxel_size_0.5.csv
    ├── {model}_voxel_size_1.0.csv
    └── ... (additional voxel sizes)
```

**OUTPUT:**
```
project_dir/
└── valid_rays/
    ├── leg_000_valid_rays.parquet   (extracted valid rays, optimized for processing)
    ├── leg_001_valid_rays.parquet
    ├── leg_002_valid_rays.parquet
    └── ... (one per leg)
```

In [ ]:
from utils import prepare_helios_data

prepare_helios_data(
    input_dir=helios_dir,
    output_dir=valid_rays_dir,
    references_dir=references_dir,
    leaf_object_ids=leaf_object_ids,
    wood_object_ids=wood_object_ids,
    use_class=use_class
)

# Step 3 - Compute Ray-Voxel Intersections
For each valid ray and voxel size, determine which voxels the ray passes through and record entry/exit points, leaf/wood classification, and other ray parameters. Output stored per-leg so metrics can be computed from different leg combinations.

In [ ]:
from utils import voxel_ray_intersections

# Run intersections
voxel_ray_intersections(
    valid_rays_dir=valid_rays_dir, 
    references_dir=references_dir,
    debug=False
)

# Step 4 - Calculate Optical Metrics and Merge with Reference Data
Aggregate ray-voxel intersections to compute per-voxel metrics (LAD, PAD, gap probability, G-function, LIAD) for selected legs and voxel sizes. Merge results with reference voxel map from `process_reference_single_obj.py` for validation.

In [ ]:
import os
import glob
import pandas as pd
from utils import get_voxel_metrics

# Select the desired legs and voxel_sizes to include in the analysis
legs = 'all'              # or specify list: [0, 1, 2]
voxel_sizes = [0.2, 0.5, 1.0, 2.0]   # or 'all'

# Set the average leaf area (m²) used for the lambda calculation
average_leaf_area = 0.003582

project_name = os.path.basename(os.path.normpath(project_dir))

# Compute per-voxel metrics. get_voxel_metrics discovers the intersection
# Parquet files in valid_rays_dir, filtered by scan (leg) and voxel size,
# computes lambda internally from average_leaf_area, writes its own per-voxel-size
# CSVs to results_dir, and returns a DataFrame spanning all processed voxel sizes.
voxel_metrics_df = get_voxel_metrics(
    intersections_folder=valid_rays_dir,
    average_leaf_area=average_leaf_area,
    output_dir=results_dir,
    project_name=project_name,
    scan_ids=None if legs == 'all' else [str(leg) for leg in legs],
    voxel_sizes=None if voxel_sizes == 'all' else voxel_sizes,
    is_multireturn=False,
)

if voxel_metrics_df is None or voxel_metrics_df.empty:
    print("No voxel metrics were computed. Please check the input parameters.")
else:
    # Merge each voxel size's metrics with its reference voxel map and save,
    # producing the validation output.
    for voxel_size, metrics in voxel_metrics_df.groupby('voxel_size', sort=False):
        metrics = metrics.reset_index(drop=True)

        reference_matches = glob.glob(os.path.join(references_dir, f'*voxel_size_{voxel_size}*'))
        if not reference_matches:
            print(f"No reference file found for voxel size {voxel_size}; skipping merge.")
            continue
        reference_file = reference_matches[0]

        # Load reference data and aggregate by voxel_id (in case of multiple entries)
        df_ref = pd.read_csv(reference_file)
        df_ref = df_ref.groupby('voxel_id').mean(numeric_only=True).reset_index()
        df_ref = df_ref.add_suffix('_ref')
        df_ref.rename(columns={
            'voxel_id_ref': 'voxel_id',
            'LAD_ref_ref': 'LAD_ref',
            'PAD_ref_ref': 'PAD_ref'
        }, inplace=True)

        # Merge HELIOS metrics with reference data
        merged = metrics.merge(df_ref, on='voxel_id', how='left')

        # Clean up duplicate coordinate columns
        merged = merged.drop(columns=[
            col for col in ['voxel_cx_ref', 'voxel_cy_ref', 'voxel_cz_ref']
            if col in merged.columns
        ])

        # Determine which legs contributed to this voxel size (for the filename)
        legs_in_files = []
        for file in glob.glob(os.path.join(valid_rays_dir, f'leg_*_voxel_{voxel_size}_intersections.parquet')):
            parts = os.path.basename(file).split('_')
            legs_in_files.append(int(parts[parts.index('leg') + 1]))
        legs_in_files = sorted(set(legs_in_files))
        leg_string = "_".join(map(str, legs_in_files))

        output_file = os.path.join(results_dir, f"{project_name}_leg_{leg_string}_voxel_size_{voxel_size}.csv")
        if os.path.exists(output_file):
            os.remove(output_file)

        merged.to_csv(output_file, index=False)
        print(f"Results saved to {output_file}")

# Utility Functions
Convert ray-voxel intersection Parquet files to CSV format for inspection or external analysis.

In [ ]:
import glob
from utils import convert_parquet_to_csv

voxel_sizes = [2.0] #[0.2, 0.5, 1.0, 2.0]
for voxel_size in voxel_sizes:
    input_files = glob.glob(os.path.join(valid_rays_dir, f'leg_*_voxel_{voxel_size}_intersections.parquet'))
    if input_files:
        for input_file in input_files:
            output_file = input_file.replace('.parquet', '.csv')
            convert_parquet_to_csv(input_file, output_file)
            print(f"Converted {input_file} to {output_file}")